
# 練習問題: ベクトル型で配列カーネルを書く

## 目標

配列の各要素に同じ計算 `y[i] = 2*x[i] + 1` を施すループを, ベクトル型を使って **`nl` 個ずつまとめて** 計算する方法を身につける. 「読み込み (load) → ベクトル演算 → 書き戻し (store)」という SIMD プログラミングの基本形を体験する.

## 課題

### C++ (ベクトル型)

`map_kernel.cpp` は, `double` を `nl=8` 個束ねたベクトル型 `doublev` を使い, 配列を `nl` 個ずつ処理する.
各かたまりについて `x[i..i+nl-1]` をベクトル `xv` として読み込み済み, 結果を `y[i..i+nl-1]` に書き戻すところも書いてある.
真ん中の **ベクトル計算が `TODO` で空** なので, 現状の `y` は不定値になり検算に失敗する.

`TODO` の箇所に **`yv = 2*x + 1` をベクトルのまま計算する1行** を書け.

```cpp
yv = 2.0 * xv + 1.0;
```

`xv` はベクトル型なので, `*` `+` は要素ごとのSIMD演算になる (スカラの `2.0`, `1.0` は自動的に全要素へ broadcast される).

### Fortran (配列演算)

ベクトル型は C/C++ 独自の拡張で Fortran には無い. `map_kernel.f90` では普通のループで `y(i) = 2*x(i) + 1` を書く (`TODO` を埋める). このような配列演算は Fortran のコンパイラが自動的にSIMD化する.

## コンパイルと実行

```
# C++
nvc++ -fast map_kernel.cpp -o map_kernel.exe

# Fortran
nvfortran -fast map_kernel.f90 -o map_kernel.exe
```

```
./map_kernel.exe 64
```

(余裕があれば `nvc++ -fast -Mkeepasm -Minfo -c map_kernel.cpp` で `.s` を見て, `vfmadd...pd` / `vmulpd` などの packed double 命令が出ていることを確認せよ.)

## 期待される結果

```
OK: y[0]=1.0, y[63]=127.0
```

`TODO` を埋める前は `y` が不定値になり `NG` と表示される (計算できていないことの確認).



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ map_kernel.cpp
#include <cstdio>
#include <cstdlib>

enum { nl = 8 };   /* double 8つ = 512 bit のベクトル長 */
typedef double doublev __attribute__((vector_size(sizeof(double) * nl)));

int main(int argc, char ** argv) {
  /* n は簡単のため nl の倍数とする */
  long n = (argc > 1 ? atol(argv[1]) : 64);
  double * x = (double *)malloc(sizeof(double) * n);
  double * y = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) x[i] = i;

  /* y[i] = 2*x[i] + 1 を, nl 個ずつまとめて (ベクトル型で) 計算する */
  for (long i = 0; i < n; i += nl) {
    doublev xv = *(doublev *)&x[i];   /* x[i..i+nl-1] を1つのベクトルとして読む */
    doublev yv;
    // TODO: xv を使い y = 2*x + 1 を「ベクトルのまま」計算して yv に求めよ.
    *(doublev *)&y[i] = yv;           /* y[i..i+nl-1] に書き戻す */
  }

  long err = 0;
  for (long i = 0; i < n; i++) if (y[i] != 2 * x[i] + 1) err++;
  if (err == 0) printf("OK: y[0]=%.1f, y[%ld]=%.1f\n", y[0], n - 1, y[n - 1]);
  else          printf("NG: %ld 要素が不正\n", err);
  free(x); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore map_kernel.cpp -o map_kernel_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./map_kernel_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ map_kernel.f90
! 注: ベクトル型 (vector_size) は C/C++ 独自の拡張で Fortran には無い.
! Fortran では普通のループ (または配列演算 y = 2*x + 1) を書けば,
! コンパイラが自動的にSIMD化してくれる.
program map_kernel
  character(len=32) :: arg
  integer(8) :: n, i, err
  real(8), allocatable :: x(:), y(:)
  n = 64
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(x(n), y(n))
  do i = 1, n
     x(i) = i - 1
  end do

  ! y(i) = 2*x(i) + 1 を計算する
  do i = 1, n
     ! TODO: y(i) = 2*x(i) + 1 を計算せよ (Fortran はこの配列演算を自動でSIMD化する).
  end do

  err = 0
  do i = 1, n
     if (y(i) /= 2.0d0 * x(i) + 1.0d0) err = err + 1
  end do
  if (err == 0) then
     print "(a,f0.1,a,i0,a,f0.1)", "OK: y(1)=", y(1), ", y(", n, ")=", y(n)
  else
     print "(a,i0,a)", "NG: ", err, " 要素が不正"
  end if
end program map_kernel

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore map_kernel.f90 -o map_kernel_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./map_kernel_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:map_kernel.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:map_kernel.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:map_kernel.cpp}